##  calibration objective : Least squares in implied vol

$$
\min_{\theta} \sum_{i,j} w_{ij}
\left(
IV_{\text{model}}(i, j; \theta) - IV_{\text{mkt}}(i, j)
\right)^2
$$

- $ w_{ij}$ weights: often $ 1/\text{spread}^2 $, or used to downweight wings / illiquid points.


In [1]:
import sys, os 
sys.path.append(os.path.abspath("../.."))


In [2]:
import plotly.graph_objects as go
import numpy as np

```text 
heston.py
│
│  → single maturity, single simulation
│
└── model_price_surface.py
    │
    │  → loops maturities × strikes
    │
    └── calibration.ipynb
        │
        │  market IV → prices
        │  fixed random numbers
        │  loss
        │  optimizer
```


In [3]:
ticker = "NVDA"

r=0.01
q=0.0

In [4]:
from surfaces.market_iv_surface import build_market_iv_surface_moneyness


spot, m_grid, maturities, Z_mkt = build_market_iv_surface_moneyness(
    
    ticker,
    m_grid=np.linspace(0.8, 1.2, 21),
    min_oi=5,
    max_expiries=50,
    interp_kind="pchip",
    use_forward=True,
    r=0.03,
    q=0.01,
)


In [5]:
from models.black76 import black76_price


def market_prices_from_iv(spot, m_grid, maturities, r, q, Z_mkt):
    P = np.full_like(Z_mkt, np.nan)
    for i, T in enumerate(maturities):
        df = np.exp(-r*T)
        F = spot * np.exp((r-q)*T)
        for j, m in enumerate(m_grid):
            K = m * F
            iv = Z_mkt[i, j]
            if np.isfinite(iv):
                P[i, j] = black76_price(F, K, T, df, iv, "C")
    return P

In [6]:
P_mkt = market_prices_from_iv(spot, m_grid, maturities, r, q, Z_mkt)
P_mkt

array([[           nan,            nan,            nan,            nan,
                   nan,            nan,            nan,            nan,
                   nan,            nan,            nan, 6.34472450e-06,
        6.83890562e-13, 5.50191296e-08, 8.67858249e-12, 1.27576959e-05,
        2.43650595e-07, 2.79984974e-09, 1.97548517e-11, 8.74082212e-14,
        1.14976526e-06],
       [           nan,            nan,            nan,            nan,
                   nan,            nan,            nan,            nan,
                   nan,            nan,            nan, 3.19459762e-10,
        4.34559306e-06, 5.57636897e-05, 1.27897991e-05, 8.73399818e-08,
        1.66456009e-07, 3.85156335e-05, 5.17808010e-05, 6.08554804e-06,
        6.11178454e-07],
       [           nan,            nan,            nan,            nan,
                   nan,            nan,            nan,            nan,
                   nan,            nan,            nan, 1.55998790e-08,
        3.5515

In [7]:
rng = np.random.default_rng(1234)

n_steps = 400
n_paths = 20000

Zs = [
    (
        rng.standard_normal((n_steps, n_paths)),
        rng.standard_normal((n_steps, n_paths))
    )
    for _ in maturities
]


### Loss definition:
- Monte Carlo prices are noisy; implied vol is a non-linear transform that amplifies noise, so we calibrate MC models on prices

In [8]:
from surfaces.model_iv_surface import heston_price_surface_mc_euler


def heston_loss_mc(
    params,
    spot, m_grid, maturities, r, q,
    P_mkt,
    Zs,
    n_steps=400,
):
    """
    Mean squared error between market prices and MC model prices
    """

    v0, kappa, theta, sigma, rho = params

    # ---- hard constraints (reject invalid params) ----
    if (
        v0 <= 0
        or kappa <= 0
        or theta <= 0
        or sigma <= 0
        or abs(rho) >= 0.999
    ):
        return 1e10

    # ---- model prices via Euler MC ----
    P_model = heston_price_surface_mc_euler(
        spot,
        m_grid,
        maturities,
        r, q,
        params,
        Zs,
        n_steps=n_steps
    )

    # ---- compare only where market data exists ----
    mask = np.isfinite(P_mkt) & np.isfinite(P_model)
    if mask.sum() < 5:
        return 1e10

    diff = P_model[mask] - P_mkt[mask]
    val = np.mean(diff**2)
    print("loss : " , val)
    return val


In [9]:
loss = lambda p: heston_loss_mc(
    p,
    spot, m_grid, maturities, r, q,
    P_mkt,
    Zs,
    n_steps=400
)


### Optimizer:

In [10]:
def progress_callback(xk, convergence):
    print(
        "params =", np.round(xk, 4),
        "| convergence =", convergence
    )
    return False  # return True would stop the optimizer


In [11]:
from scipy.optimize import differential_evolution

bounds = [
    (1e-4, 1.0),     # v0
    (1e-3, 20.0),    # kappa
    (1e-4, 1.0),     # theta
    (1e-3, 3.0),     # sigma
    (-0.999, 0.999), # rho
]

res = differential_evolution(
    heston_loss_mc,
    bounds=bounds,
    args=(spot, m_grid, maturities, r, q, P_mkt, Zs, n_steps),
    maxiter=10,
    popsize=8,
    seed=1,
    callback=progress_callback,
    disp=True
)

res.x


loss :  2263.0610553917572
loss :  1697.0575642113097
loss :  1743.4392344612486
loss :  3199.5891794982886
loss :  1547.467853598227
loss :  1545.6459288171789
loss :  11.487478027336108
loss :  1569.0598778938775
loss :  1157.124872603792
loss :  2709.4379883667903
loss :  2131.532739371458
loss :  670.9128411780679
loss :  566.0160715863228
loss :  1236.00039903012
loss :  928.4820944210123
loss :  2311.638703283295
loss :  266.4328302496378
loss :  2830.0508660946066
loss :  442.92733844183715
loss :  1922.389022637513
loss :  1284.8459066051191
loss :  858.5832408700405
loss :  512.5085556049193
loss :  596.7125221402664
loss :  1201.943924188753
loss :  2567.6377206854145
loss :  2575.452576798877
loss :  399.93293897082395
loss :  1817.2977318960186
loss :  610.047412826133
loss :  1256.3444334445003
loss :  453.217452232611
loss :  477.33724438205365
loss :  2217.0377766289243
loss :  556.347260368561
loss :  1868.9935801894642
loss :  840.780896022896
loss :  2676.195184489222

KeyboardInterrupt: 

- total eval = maxiter × popsize × dimension

- (# maturities)\
× (1 Euler simulation per maturity)\
× (n_steps × n_paths)\
× (pricing all strikes)

last result : array([ 2.12916847e-03,  1.80610769e+01,  1.00000000e-04,  1.42867754e-01,
       -9.75515245e-01])


In [ ]:
# import QuantLib as ql
# print(ql.__version__)


In [ ]:

# def calibrate_heston_to_iv_mgrid(
#     spot, m_grid, maturities, r, q, Z_mkt,
#     v0=0.04, kappa=2.0, theta=0.04, sigma=0.5, rho=-0.6,
#     method="lm"
# ):

#     day_count = ql.Actual365Fixed()
#     cal = ql.NullCalendar()
#     today = ql.Date.todaysDate()
#     ql.Settings.instance().evaluationDate = today

#     spot_q = ql.SimpleQuote(float(spot))
#     r_ts = ql.FlatForward(today, ql.QuoteHandle(ql.SimpleQuote(float(r))), day_count)
#     q_ts = ql.FlatForward(today, ql.QuoteHandle(ql.SimpleQuote(float(q))), day_count)

#     process = ql.HestonProcess(
#         ql.YieldTermStructureHandle(r_ts),
#         ql.YieldTermStructureHandle(q_ts),
#         ql.QuoteHandle(spot_q),
#         float(v0), float(kappa), float(theta), float(sigma), float(rho)
#     )
#     model = ql.HestonModel(process)
#     engine = ql.AnalyticHestonEngine(model)

#     # ----- one per (T, K) quote -----
#     helpers = []
#     for i, T in enumerate(maturities):
#         if T <= 0: 
#             continue

#         # maturity date from year fraction (rough, good enough for equity options)
#         days = int(round(T * 365))
#         maturity_date = today + days
#         period = ql.Period(days, ql.Days)

#         F = spot * np.exp((r - q) * T)
#         for j, m in enumerate(m_grid):
#             iv = Z_mkt[i, j]
#             if not np.isfinite(iv) or iv <= 0:
#                 continue

#             K = float(m * F)

#             vol_quote = ql.SimpleQuote(float(iv))
#             helper = ql.HestonModelHelper(
#                 period, cal,
#                 float(spot), K,
#                 ql.QuoteHandle(vol_quote),
#                 ql.YieldTermStructureHandle(r_ts),
#                 ql.YieldTermStructureHandle(q_ts),
#                 ql.BlackCalibrationHelper.ImpliedVolError
#             )
#             helper.setPricingEngine(engine)
#             helpers.append(helper)

#     # ----- optimizer -----
#     if method == "lm":
#         opt = ql.LevenbergMarquardt()
#     else:
#         opt = ql.Simplex(0.1)

#     end_crit = ql.EndCriteria(500, 50, 1e-8, 1e-8, 1e-8)

#     model.calibrate(helpers, opt, end_crit)

#     # calibrated params
#     p = model.params()  # [theta, kappa, sigma, rho, v0] ordering in QL is different in places
#     # safest: read from process/model directly:
#     return {
#         "v0": float(model.v0()),
#         "kappa": float(model.kappa()),
#         "theta": float(model.theta()),
#         "sigma": float(model.sigma()),
#         "rho": float(model.rho()),
#         "calib_error": float(sum(h.calibrationError()**2 for h in helpers) / max(len(helpers),1))
#     }


In [ ]:
# params_star = calibrate_heston_to_iv_mgrid(
#     spot, m_grid, maturities, r, q, Z_mkt
# )
# params_star
